In [51]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest


In [52]:
dataset = pd.read_csv('../outputs/tables/vsDataEUtranCellRelation.csv')
dataset.columns

Index(['DN', 'id', 'vsDataType', 'timeOfCreation', 'lastModification',
       'isRemoveAllowed', 'lbBnrAllowed', 'timeOfLastModification',
       'qOffsetCellEUtran', 'sCellCandidate', 'isHoAllowed',
       'cellIndividualOffsetEUtran', 'coverageIndicator',
       'incomingLoadBalancing', 'includeInSystemInformation', 'lbCovIndicated',
       'loadBalancing', 'createdBy', 'adjacentCell',
       'sleepModeCovCellCandidate', 'sleepModeCoverageCell',
       'sleepModeCapacityCell', 'hoSuccLevel', 'mobilityStatus.available',
       'crsAssistanceInfoPriority', 'amoAllowed', 'amoState',
       'mobilityStatus.reason'],
      dtype='object')

In [53]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 38 entries, 0 to 37
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   DN                          38 non-null     object 
 1   id                          38 non-null     object 
 2   vsDataType                  38 non-null     object 
 3   timeOfCreation              37 non-null     object 
 4   lastModification            38 non-null     int64  
 5   isRemoveAllowed             38 non-null     bool   
 6   lbBnrAllowed                38 non-null     bool   
 7   timeOfLastModification      6 non-null      object 
 8   qOffsetCellEUtran           38 non-null     int64  
 9   sCellCandidate              38 non-null     int64  
 10  isHoAllowed                 38 non-null     bool   
 11  cellIndividualOffsetEUtran  38 non-null     int64  
 12  coverageIndicator           38 non-null     int64  
 13  incomingLoadBalancing       38 non-nu

In [54]:
dataset['mobilityStatus.available'].value_counts()

mobilityStatus.available
True     36
False     2
Name: count, dtype: int64

In [55]:
COLONNES_A_RETIRER=[
    'id' ,'vsDataType' ,'adjacentCell','timeOfCreation','timeOfLastModification','lastModification','createdBy','mobilityStatus.reason'
]
X = dataset.set_index('DN').drop(columns=COLONNES_A_RETIRER,errors='ignore'
)

In [56]:
X=X.astype(float)
X.head()

,isRemoveAllowed,lbBnrAllowed,qOffsetCellEUtran,sCellCandidate,isHoAllowed,cellIndividualOffsetEUtran,coverageIndicator,incomingLoadBalancing,includeInSystemInformation,lbCovIndicated,loadBalancing,sleepModeCovCellCandidate,sleepModeCoverageCell,sleepModeCapacityCell,hoSuccLevel,mobilityStatus.available,crsAssistanceInfoPriority,amoAllowed,amoState
DN,,,,,,,,,,,,,,,,,,,
"SubNetwork=ONRM_ROOT_MO_R,SubNetwork=RadioNodes,MeContext=AHO-1001_BB_L,ManagedElement=AHO-1001_BB_L,vsDataENodeBFunction=1,vsDataEUtranCellFDD=AHO-1001_L-1,vsDataEUtranFreqRelation=1650,vsDataEUtranCellRelation=6042-112146-2",1.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,2.0,1.0,0.0,1.0,1.0
"SubNetwork=ONRM_ROOT_MO_R,SubNetwork=RadioNodes,MeContext=AHO-1001_BB_L,ManagedElement=AHO-1001_BB_L,vsDataENodeBFunction=1,vsDataEUtranCellFDD=AHO-1001_L-1,vsDataEUtranFreqRelation=1650,vsDataEUtranCellRelation=6042-126001-2",0.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
"SubNetwork=ONRM_ROOT_MO_R,SubNetwork=RadioNodes,MeContext=AHO-1001_BB_L,ManagedElement=AHO-1001_BB_L,vsDataENodeBFunction=1,vsDataEUtranCellFDD=AHO-1001_L-1,vsDataEUtranFreqRelation=1650,vsDataEUtranCellRelation=6042-126001-3",0.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
"SubNetwork=ONRM_ROOT_MO_R,SubNetwork=RadioNodes,MeContext=AHO-1001_BB_L,ManagedElement=AHO-1001_BB_L,vsDataENodeBFunction=1,vsDataEUtranCellFDD=AHO-1001_L-1,vsDataEUtranFreqRelation=1650,vsDataEUtranCellRelation=6042-126002-1",1.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0
"SubNetwork=ONRM_ROOT_MO_R,SubNetwork=RadioNodes,MeContext=AHO-1001_BB_L,ManagedElement=AHO-1001_BB_L,vsDataENodeBFunction=1,vsDataEUtranCellFDD=AHO-1001_L-1,vsDataEUtranFreqRelation=1650,vsDataEUtranCellRelation=6042-126002-2",1.0,1.0,0.0,2.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,2.0,0.0,0.0,1.0,1.0,0.0,1.0,1.0


In [57]:
model = IsolationForest(contamination=0.1,random_state=42)
predictions = model.fit_predict(X)
scores =model.decision_function(X)

X['anomalie']=predictions
X['score']=scores

In [58]:
X['anomalie'].value_counts()

anomalie
 1    36
-1     2
Name: count, dtype: int64

In [59]:
colonnes_reglages = X.columns.drop(['anomalie','score'])
normaux = X[X['anomalie']==1]
anormaux = X[X['anomalie']==-1]

valeurs_normales = normaux[colonnes_reglages].mode().iloc[0]
for  id ,  ligne in anormaux.iterrows():
    print(f"\nAnomalie détectée :")
    print(f"  voisin : {id}")
    print(f"  score  : {ligne['score']:.3f}")
    print(f"  colonnes déviantes :")
    for col in colonnes_reglages:
        valeur_voisin = ligne[col]
        valeur_normale = valeurs_normales[col]
        if valeur_voisin != valeur_normale:
            print(f"    - {col} = {valeur_voisin}  (majorité : {valeur_normale})")




Anomalie détectée :
  voisin : SubNetwork=ONRM_ROOT_MO_R,SubNetwork=RadioNodes,MeContext=AHO-1001_BB_L,ManagedElement=AHO-1001_BB_L,vsDataENodeBFunction=1,vsDataEUtranCellFDD=AHO-1001_L-1,vsDataEUtranFreqRelation=1650,vsDataEUtranCellRelation=6042-126004-1
  score  : -0.151
  colonnes déviantes :
    - hoSuccLevel = 3.0  (majorité : 1.0)
    - mobilityStatus.available = 0.0  (majorité : 1.0)

Anomalie détectée :
  voisin : SubNetwork=ONRM_ROOT_MO_R,SubNetwork=RadioNodes,MeContext=AHO-1001_BB_L,ManagedElement=AHO-1001_BB_L,vsDataENodeBFunction=1,vsDataEUtranCellFDD=AHO-1001_L-2,vsDataEUtranFreqRelation=1650,vsDataEUtranCellRelation=6042-126004-1
  score  : -0.151
  colonnes déviantes :
    - hoSuccLevel = 3.0  (majorité : 1.0)
    - mobilityStatus.available = 0.0  (majorité : 1.0)
